# 01 — ProductPulse Dataset Inventory

Goal: inspect the four raw public datasets **without cleaning or transforming them**.

We will record:
- file sizes
- row/column counts where practical
- column names and dtypes
- missingness
- date ranges
- key categorical values

Datasets:
1. Retailrocket
2. Cookie Cats
3. Criteo Uplift
4. Online Retail II

In [32]:
from pathlib import Path

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for p in candidates:
        if (p / "data" / "raw").exists():
            return p
    # Common local location used for this project
    fallback = Path.home() / "ProductPulse"
    if (fallback / "data" / "raw").exists():
        return fallback
    raise FileNotFoundError("Could not locate ProductPulse project root.")

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)

PROJECT_ROOT: /Users/noopur/Desktop/resume_projects/ProductPulse
RAW_DIR: /Users/noopur/Desktop/resume_projects/ProductPulse/data/raw


In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## File inventory

In [34]:
records = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_file():
        records.append({
            "dataset": path.parent.name,
            "file": path.name,
            "size_mb": round(path.stat().st_size / (1024**2), 2)
        })

files_df = pd.DataFrame(records)
files_df

,dataset,file,size_mb
0,cookie_cats,cookie_cats.csv,2.58
1,criteo,criteo-uplift-v2.1.csv.gz,297.00
2,online_retail,online_retail_II.xlsx,43.51
3,online_retail,online_retail_ii.zip,43.51
4,retailrocket,category_tree.csv,0.01
5,retailrocket,ecommerce-dataset.zip,290.60
6,retailrocket,events.csv,89.87
7,retailrocket,item_properties_part1.csv,461.88
8,retailrocket,item_properties_part2.csv,389.99


## Retailrocket — events

In [35]:
rr_events_path = RAW_DIR / "retailrocket" / "events.csv"
rr = pd.read_csv(rr_events_path)

print("Shape:", rr.shape)
print("\nColumns:", rr.columns.tolist())
display(rr.head())
display(rr.dtypes.to_frame("dtype"))
display(rr.isna().mean().sort_values(ascending=False).to_frame("missing_pct"))

Shape: (2756101, 5)

Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


,dtype
timestamp,int64
visitorid,int64
event,str
itemid,int64
transactionid,float64


,missing_pct
transactionid,0.991852
timestamp,0.000000
visitorid,0.000000
event,0.000000
itemid,0.000000


In [36]:
rr["event_time"] = pd.to_datetime(rr["timestamp"], unit="ms", errors="coerce")
print("Date range:", rr["event_time"].min(), "to", rr["event_time"].max())
display(rr["event"].value_counts(dropna=False).to_frame("rows"))
print("Unique visitors:", rr["visitorid"].nunique())
print("Unique items:", rr["itemid"].nunique())

Date range: 2015-05-03 03:00:04.384000 to 2015-09-18 02:59:47.788000


,rows
event,
view,2664312
addtocart,69332
transaction,22457


Unique visitors: 1407580
Unique items: 235061


### Retailrocket product metadata

The item-property files are much larger than the event file. We inspect only a sample here so the notebook stays lightweight.

In [37]:
rr_prop1_path = RAW_DIR / "retailrocket" / "item_properties_part1.csv"
rr_prop2_path = RAW_DIR / "retailrocket" / "item_properties_part2.csv"
rr_cat_path = RAW_DIR / "retailrocket" / "category_tree.csv"

prop1_sample = pd.read_csv(rr_prop1_path, nrows=100_000)
prop2_sample = pd.read_csv(rr_prop2_path, nrows=100_000)
cat = pd.read_csv(rr_cat_path)

print("Property sample columns:", prop1_sample.columns.tolist())
print("Category tree shape:", cat.shape)
display(prop1_sample.head())
display(cat.head())

Property sample columns: ['timestamp', 'itemid', 'property', 'value']
Category tree shape: (1669, 2)


,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


## Cookie Cats

In [38]:
cc_path = RAW_DIR / "cookie_cats" / "cookie_cats.csv"
cc = pd.read_csv(cc_path)

print("Shape:", cc.shape)
display(cc.head())
display(cc.dtypes.to_frame("dtype"))
display(cc.isna().mean().sort_values(ascending=False).to_frame("missing_pct"))
display(cc["version"].value_counts(dropna=False).to_frame("users"))

Shape: (90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


,dtype
userid,int64
version,str
sum_gamerounds,int64
retention_1,bool
retention_7,bool


,missing_pct
userid,0.0
version,0.0
sum_gamerounds,0.0
retention_1,0.0
retention_7,0.0


,users
version,
gate_40,45489
gate_30,44700


## Criteo Uplift — sample + chunked row count

In [39]:
criteo_path = RAW_DIR / "criteo" / "criteo-uplift-v2.1.csv.gz"

criteo_sample = pd.read_csv(criteo_path, compression="gzip", nrows=100_000)
print("Sample shape:", criteo_sample.shape)
print("Columns:", criteo_sample.columns.tolist())
display(criteo_sample.head())
display(criteo_sample.dtypes.to_frame("dtype"))
display(criteo_sample.isna().mean().sort_values(ascending=False).to_frame("missing_pct"))

Sample shape: (100000, 16)
Columns: ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


,dtype
f0,float64
f1,float64
f2,float64
f3,float64
f4,float64
f5,float64
f6,float64
f7,float64
f8,float64
f9,float64


,missing_pct
f0,0.0
f1,0.0
f2,0.0
f3,0.0
f4,0.0
f5,0.0
f6,0.0
f7,0.0
f8,0.0
f9,0.0


In [40]:
# Count rows without loading all ~14M rows into memory.
criteo_rows = 0
for chunk in pd.read_csv(criteo_path, compression="gzip", chunksize=500_000):
    criteo_rows += len(chunk)

print("Criteo total rows:", f"{criteo_rows:,}")

Criteo total rows: 13,979,592


## Online Retail II

In [41]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [42]:
retail_path = RAW_DIR / "online_retail" / "online_retail_II.xlsx"
xls = pd.ExcelFile(retail_path)

print("Sheets:", xls.sheet_names)

Sheets: ['Year 2009-2010', 'Year 2010-2011']


In [43]:
retail_frames = []
for sheet in xls.sheet_names:
    tmp = pd.read_excel(retail_path, sheet_name=sheet)
    tmp["source_sheet"] = sheet
    retail_frames.append(tmp)

online = pd.concat(retail_frames, ignore_index=True)

print("Shape:", online.shape)
print("Columns:", online.columns.tolist())
display(online.head())
display(online.dtypes.to_frame("dtype"))
display(online.isna().mean().sort_values(ascending=False).to_frame("missing_pct"))

Shape: (1067371, 9)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


,dtype
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str
source_sheet,str


,missing_pct
Customer ID,0.227669
Description,0.004105
Invoice,0.000000
StockCode,0.000000
Quantity,0.000000
InvoiceDate,0.000000
Price,0.000000
Country,0.000000
source_sheet,0.000000


In [44]:
# Identify date column robustly
date_col = next((c for c in online.columns if str(c).lower() in {"invoicedate", "invoice_date"}), None)
if date_col:
    online[date_col] = pd.to_datetime(online[date_col], errors="coerce")
    print("Date range:", online[date_col].min(), "to", online[date_col].max())

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


## Final inventory summary

In [45]:
summary = pd.DataFrame([
    {
        "dataset": "Retailrocket",
        "type": "Behavioral events",
        "approx_rows": len(rr),
        "primary_entity": "visitorid",
        "use": "funnels, sessions, conversion, repeat behavior"
    },
    {
        "dataset": "Cookie Cats",
        "type": "Randomized experiment",
        "approx_rows": len(cc),
        "primary_entity": "userid",
        "use": "A/B testing, retention, engagement"
    },
    {
        "dataset": "Criteo Uplift",
        "type": "Treatment/control experiment",
        "approx_rows": criteo_rows,
        "primary_entity": "row/user",
        "use": "treatment effects, uplift, targeting"
    },
    {
        "dataset": "Online Retail II",
        "type": "Transactions",
        "approx_rows": len(online),
        "primary_entity": "customer/invoice",
        "use": "revenue, cohorts, repeat purchase, customer analytics"
    },
])

summary

,dataset,type,approx_rows,primary_entity,use
0,Retailrocket,Behavioral events,2756101,visitorid,"funnels, sessions, conversion, repeat behavior"
1,Cookie Cats,Randomized experiment,90189,userid,"A/B testing, retention, engagement"
2,Criteo Uplift,Treatment/control experiment,13979592,row/user,"treatment effects, uplift, targeting"
3,Online Retail II,Transactions,1067371,customer/invoice,"revenue, cohorts, repeat purchase, customer an..."


In [46]:
# ============================================================
# SAVE NOTEBOOK 01 RESULTS ONLY
# Product Pulse
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Project / results paths
# ------------------------------------------------------------



PROJECT_ROOT = Path.home() / "Desktop" / "resume_projects" / "ProductPulse"

RESULTS_DIR = PROJECT_ROOT / "results" / "01_data_inventory_and_profiling"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Results dir :", RESULTS_DIR)


# ------------------------------------------------------------
# 2. Save ONLY small summary/result tables
# ------------------------------------------------------------

MAX_RESULT_ROWS = 500
MAX_RESULT_COLS = 100

saved_tables = []

# Names that usually indicate raw/full datasets.
# These will NEVER be saved.
EXCLUDE_NAMES = {
    "df",
    "data",
    "raw",
    "retail",
    "retail_df",
    "online_retail",
    "transactions",
    "orders",
    "products",
    "customers",
}

for name, obj in list(globals().items()):

    if name.startswith("_"):
        continue

    if name.lower() in EXCLUDE_NAMES:
        continue

    # -------------------------
    # DataFrames
    # -------------------------
    if isinstance(obj, pd.DataFrame):

        rows, cols = obj.shape

        if rows <= MAX_RESULT_ROWS and cols <= MAX_RESULT_COLS:

            output_path = TABLES_DIR / f"{name}.csv"

            obj.to_csv(
                output_path,
                index=True
            )

            saved_tables.append({
                "name": name,
                "type": "DataFrame",
                "rows": rows,
                "columns": cols
            })

            print(
                f"Saved table: {name} "
                f"({rows:,} rows × {cols:,} cols)"
            )

    # -------------------------
    # Series
    # -------------------------
    elif isinstance(obj, pd.Series):

        if len(obj) <= MAX_RESULT_ROWS:

            output_path = TABLES_DIR / f"{name}.csv"

            obj.to_csv(
                output_path,
                index=True
            )

            saved_tables.append({
                "name": name,
                "type": "Series",
                "rows": len(obj),
                "columns": 1
            })

            print(
                f"Saved series: {name} "
                f"({len(obj):,} rows)"
            )


# ------------------------------------------------------------
# 3. Save currently open matplotlib figures
# ------------------------------------------------------------

saved_figures = []

for i, fig_num in enumerate(plt.get_fignums(), start=1):

    fig = plt.figure(fig_num)

    output_path = FIGURES_DIR / f"figure_{i:02d}.png"

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    saved_figures.append(output_path.name)

    print(f"Saved figure: {output_path.name}")


# ------------------------------------------------------------
# 4. Save a small manifest
# ------------------------------------------------------------

manifest = pd.DataFrame(saved_tables)

if not manifest.empty:
    manifest.to_csv(
        RESULTS_DIR / "tables_manifest.csv",
        index=False
    )


# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PRODUCT PULSE — NOTEBOOK 01 RESULTS")
print("=" * 60)

print(f"Tables saved : {len(saved_tables)}")
print(f"Figures saved: {len(saved_figures)}")

print("\nSaved to:")
print(RESULTS_DIR)

Project root: /Users/noopur/Desktop/resume_projects/ProductPulse
Results dir : /Users/noopur/Desktop/resume_projects/ProductPulse/results/01_data_inventory_and_profiling
Saved table: files_df (9 rows × 3 cols)
Saved table: summary (4 rows × 5 cols)
Saved table: manifest (3 rows × 4 cols)

PRODUCT PULSE — NOTEBOOK 01 RESULTS
Tables saved : 3
Figures saved: 0

Saved to:
/Users/noopur/Desktop/resume_projects/ProductPulse/results/01_data_inventory_and_profiling


## Notebook conclusion

Do not harmonize schemas here. This notebook is only the raw-data inventory.
The reusable canonical schema belongs in the ProductPulse infrastructure layer.